In [603]:
# initialize stuff
import sys
import importlib

from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import superconductivity.api as sc

from IPython import get_ipython

_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic("matplotlib", "inline")
    _ip.run_line_magic("reload_ext", "autoreload")
    _ip.run_line_magic("autoreload", "2")

    _ip.run_line_magic(
        "config",
        "InlineBackend.print_figure_kwargs = {'bbox_inches': None, 'pad_inches': 0.0}",
    )
    _ip.run_line_magic("config", 'InlineBackend.figure_format = "retina"')  # or "png"
    _ip.run_line_magic(
        "config", "InlineBackend.rc = {'figure.dpi': 300}"
    )  # choose a value you like

In [604]:
tau = np.array([0.78, 0.66, 0.31, 0.16, 0.08, 0.03])
GN = np.sum(tau)

In [605]:
from superconductivity.evaluation import list_specific_keys_and_values
from superconductivity.evaluation.keys import list_measurement_keys

importlib.reload(sys.modules["superconductivity.evaluation"])

h5path = "data/2023-11-04_G0_antenna.hdf5"
h5path = "data/2023-11-03_G0_stripline.hdf5"
measurement = "frequency_at_7.8GHz"
mkey = list_measurement_keys(h5path=h5path)
keys, P_out_dBm = list_specific_keys_and_values(
    h5path=h5path,
    measurement=measurement,
    strip0="=",
    strip1="dBm",
)

In [606]:
from superconductivity.evaluation import get_ivt

importlib.reload(sys.modules["superconductivity.evaluation"])

V_mV_raw = []
I_nA_raw = []
t_s_raw = []

for key in keys:
    I_nA, V_mV, t_s = get_ivt(
        h5path=h5path,
        measurement=measurement,
        specific_key=key,
        amp_voltage=10000,
        amp_current=1000,
        trigger_values=1,
        skip=5,
        time_relative=True,
    )

    I_nA_raw.append(I_nA)
    V_mV_raw.append(V_mV)
    t_s_raw.append(t_s)

In [607]:
from superconductivity.evaluation import get_psd

importlib.reload(sys.modules["superconductivity.evaluation"])

Ipsd_raw = []
Vpsd_raw = []
f_Hz_raw = []

t_max_s = 0.0

for i, key in enumerate(keys):

    t_max_s = max(t_max_s, float(np.nanmax(t_s)))

    Ipsd, Vpsd, f_Hz = get_psd(
        I_nA=I_nA_raw[i],
        V_mV=V_mV_raw[i],
        t_s=t_s_raw[i],
        detrend=False,
    )

    Ipsd_raw.append(Ipsd)
    Vpsd_raw.append(Vpsd)
    f_Hz_raw.append(f_Hz)

In [608]:
title = "0 raw"
dataset = "visuals"

from superconductivity.visuals.plotly import get_tuple_slider

importlib.reload(sys.modules["superconductivity.visuals.plotly"])

fig_raw_iv = get_tuple_slider(
    x_list=V_mV_raw,
    y_list=I_nA_raw,
    slider_label="<i>P</i> (dBm)",
    xlabel="<i>V</i> (mV)",
    ylabel="<i>I</i> (nA)",
    slider_values=P_out_dBm,
    name=f"{title} iv 0",  # saves htmls if set
    dataset=dataset,
)

fig_raw_Vpsd = get_tuple_slider(
    x_list=f_Hz_raw,
    y_list=Vpsd_raw,
    slider_label="<i>P</i> (dBm)",
    xlabel="<i>f</i> (Hz)",
    ylabel=f"<i>V</i><sub>psd</sub> (mV<sup>2</sup>/Hz)",
    slider_values=P_out_dBm,
    ylogscale=True,
    name=f"{title} V(f) 1",  # saves htmls if set
    dataset=dataset,
)

fig_raw_Ipsd = get_tuple_slider(
    x_list=f_Hz_raw,
    y_list=Ipsd_raw,
    slider_label="<i>P</i> (dBm)",
    xlabel="<i>f</i> (Hz)",
    ylabel=f"<i>I</i><sub>psd</sub> (nA<sup>2</sup>/Hz)",
    slider_values=P_out_dBm,
    ylogscale=True,
    name=f"{title} I(f) 2",  # saves htmls if set
    dataset=dataset,
)

In [609]:
nu_sample_Hz = 37
dt_sample_s = 1 / (2 * nu_sample_Hz)

V_mV_down = []
I_nA_down = []
t_s_down = []

Ipsd_down = []
Vpsd_down = []
f_Hz_down = []

for i, key in enumerate(keys):
    t_max_s = float(np.nanmax(t_s_raw[i]))
    t_s = np.arange(0.0, t_max_s + dt_sample_s, dt_sample_s)

    It_nA = sc.bin_y_over_x(t_s_raw[i], I_nA_raw[i], t_s)
    Vt_mV = sc.bin_y_over_x(t_s_raw[i], V_mV_raw[i], t_s)

    mask = np.where(np.isnan(It_nA), False, True)
    mask = np.where(np.isfinite(It_nA), mask, False)

    Ipsd, Vpsd, f_Hz = get_psd(
        I_nA=It_nA[mask],
        V_mV=Vt_mV[mask],
        t_s=t_s[mask],
    )

    V_mV_down.append(Vt_mV)
    I_nA_down.append(It_nA)
    t_s_down.append(t_s)

    Ipsd_down.append(Ipsd)
    Vpsd_down.append(Vpsd)
    f_Hz_down.append(f_Hz)

In [610]:
title = "1 down"
dataset = "visuals"

from superconductivity.visuals.plotly import get_tuple_slider

importlib.reload(sys.modules["superconductivity.visuals.plotly"])

fig_raw_Vpsd = get_tuple_slider(
    x_list=V_mV_down,
    y_list=I_nA_down,
    slider_label="<i>P</i> (dBm)",
    xlabel="<i>V</i> (mV)",
    ylabel="<i>I</i> (nA)",
    slider_values=P_out_dBm,
    name=f"{title} sampled I(V) 0",  # saves htmls if set
    dataset=dataset,
)

fig_raw_Vpsd = get_tuple_slider(
    x_list=f_Hz_down,
    y_list=Vpsd_down,
    slider_label="<i>P</i> (dBm)",
    xlabel="<i>f</i> (Hz)",
    ylabel=f"<i>V</i><sub>psd</sub> (mV<sup>2</sup>/Hz)",
    slider_values=P_out_dBm,
    ylogscale=True,
    name=f"{title} sampled V(f) 1",  # saves htmls if set
    dataset=dataset,
)

fig_raw_Ipsd = get_tuple_slider(
    x_list=f_Hz_down,
    y_list=Ipsd_down,
    slider_label="<i>P</i> (dBm)",
    xlabel="<i>f</i> (Hz)",
    ylabel=f"<i>I</i><sub>psd</sub> (nA<sup>2</sup>/Hz)",
    slider_values=P_out_dBm,
    ylogscale=True,
    name=f"{title} sampled I(f) 2",  # saves htmls if set
    dataset=dataset,
)

In [611]:
from superconductivity.evaluation import get_offset

importlib.reload(sys.modules["superconductivity.evaluation"])

Voffrange_mV = np.linspace(-0.02, 0.02, 51)
Ioffrange_nA = np.linspace(-2.5, 2.5, 101)

Vbias_mV = np.linspace(-1, 1, 51)
Ibias_nA = np.linspace(-180, 180, 181)

offset = get_offset(
    v_list_mV=V_mV_down,
    i_list_nA=I_nA_down,
    V_mV=Vbias_mV,
    I_nA=Ibias_nA,
    V_off_range_mV=Voffrange_mV,
    I_off_range_nA=Ioffrange_nA,
    upsample=10,
)

get_offset:   0%|          | 0/22 [00:00<?, ?curve/s]

In [612]:
title = "2 Voff"
dataset = "visuals"

from superconductivity.visuals.plotly import get_all

figs_voff = get_all(
    x=Voffrange_mV * 1e3,
    y=P_out_dBm,
    z=offset["dGerr_G0"],
    xlabel="<i>V</i><sub>off</sub> (µV)",
    ylabel="<i>P</i> (dBm)",
    zlabel="|d<i>G(+V)</i>-d<i>G(-V)</i>| (G<sub>0</sub>)",
    name=f"{title}",
    dataset=dataset,
    show_scale=False,
)

In [613]:
title = "3 Ioff"
dataset = "visuals"

from superconductivity.visuals.plotly import get_all

figs_voff = get_all(
    x=Ioffrange_nA,
    y=P_out_dBm,
    z=offset["dRerr_R0"],
    xlabel="<i>I</i><sub>off</sub> (nA)",
    ylabel="<i>P</i> (dBm)",
    zlabel="|d<i>R(+I)</i>-d<i>R(-I)</i>| (R<sub>0</sub>)",
    name=f"{title}",
    dataset=dataset,
    show_scale=False,
)

In [614]:
# binning
ylen = P_out_dBm.shape[0]

Vbias_mV = np.linspace(-0.8, 0.8, 1001)
Ibias_nA = np.linspace(-160, 160, 1001)

Vexp_mV = np.full((ylen, Ibias_nA.shape[0]), np.nan)
Iexp_nA = np.full((ylen, Vbias_mV.shape[0]), np.nan)

for i, key in enumerate(keys):
    _i_nA = I_nA_down[i] - offset["Ioff_nA"][i]
    _v_mV = V_mV_down[i] - offset["Voff_mV"][i]

    _i_nA, _v_mV = sc.upsample(_i_nA, _v_mV)

    vexp_mV = sc.bin_y_over_x(_i_nA, _v_mV, Ibias_nA)
    iexp_nA = sc.bin_y_over_x(_v_mV, _i_nA, Vbias_mV)

    Vexp_mV[i, :] = vexp_mV
    Iexp_nA[i, :] = iexp_nA

dGexp_G0 = np.gradient(Iexp_nA, Vbias_mV, axis=1) / sc.G0_muS
dRexp_R0 = np.gradient(Vexp_mV, Ibias_nA, axis=1) * sc.G0_muS

In [615]:
title = "4 Iexp"
dataset = "visuals"

from superconductivity.visuals.plotly import get_all

importlib.reload(sys.modules["superconductivity.visuals.plotly"])

fig_surface, fig_slider, fig_heatmap = get_all(
    x=Vbias_mV,
    y=P_out_dBm,
    z=Iexp_nA,
    xlabel="<i>V</i> (mV)",
    ylabel="<i>P</i> (dBm)",
    zlabel="<i>I</i> (nA)",
    name=title,
    dataset=dataset,
)

from superconductivity.visuals.stl import write_3D_print

write_3D_print(
    z=Iexp_nA,
    title=title,
    dataset=dataset,
)

from superconductivity.visuals.svg import save_axis

axis = save_axis(
    values=Vbias_mV,
    label=r"$V\ (\mathrm{mV})$",
    title=f"x {title}",
    dataset=dataset,
)
axis = save_axis(
    values=P_out_dBm,
    label=r"$P_\mathrm{out}\ (\mathrm{dBm})$",
    title=f"y {title}",
    dataset=dataset,
    vertical=True,
)
axis = save_axis(
    values=Iexp_nA * 1e-3,
    label=r"$I\ (\mu\mathrm{A})$",
    title=f"z {title}",
    dataset=dataset,
)
axis = save_axis(
    values=Iexp_nA * 1e-3,
    label=r"$I\ (\mu\mathrm{A})$",
    title=f"z {title}2",
    dataset=dataset,
    vertical=True,
)

from superconductivity.visuals.luanti import export_dataset

export_dataset(
    Iexp_nA,
    title=title,
    dataset=dataset,
    zlim=(-150, 150),
    hmin=30,
    hmax=270,
    ylen=124,
    xlen=500,
)

PosixPath('visuals/datasets/4 Iexp/map.u16le')

In [616]:
title = "5 Vexp"
dataset = "visuals"

from superconductivity.visuals.plotly import get_all

importlib.reload(sys.modules["superconductivity.visuals.plotly"])

fig_surface, fig_slider, fig_heatmap = get_all(
    x=Ibias_nA,
    y=P_out_dBm,
    z=Vexp_mV,
    xlabel="<i>I</i> (nA)",
    ylabel="<i>P</i> (dBm)",
    zlabel="<i>V</i> (mV)",
    name=title,
    dataset=dataset,
)

from superconductivity.visuals.stl import write_3D_print

write_3D_print(
    z=Vexp_mV,
    title=title,
    dataset=dataset,
)

from superconductivity.visuals.svg import save_axis

axis = save_axis(
    values=Ibias_nA * 1e-3,
    label=r"$I\ (\mu\mathrm{A})$",
    title=f"x {title}",
    dataset=dataset,
)
axis = save_axis(
    values=P_out_dBm,
    label=r"$P_\mathrm{out}\ (\mathrm{dBm})$",
    title=f"y {title}",
    dataset=dataset,
    vertical=True,
)
axis = save_axis(
    values=Vexp_mV,
    label=r"$V\ (\mathrm{mV})$",
    title=f"z {title}",
    dataset=dataset,
)
axis = save_axis(
    values=Vexp_mV,
    label=r"$V\ (\mathrm{mV})$",
    title=f"z {title}2",
    dataset=dataset,
    vertical=True,
)

from superconductivity.visuals.luanti import export_dataset

export_dataset(
    Vexp_mV,
    title=title,
    dataset=dataset,
    zlim=(-0.8, 0.8),
    hmin=30,
    hmax=270,
    ylen=124,
    xlen=500,
)

PosixPath('visuals/datasets/5 Vexp/map.u16le')

In [617]:
title = "6 dGexp"
dataset = "visuals"

from superconductivity.visuals.plotly import get_all

importlib.reload(sys.modules["superconductivity.visuals.plotly"])

fig_surface, fig_slider, fig_heatmap = get_all(
    x=Vbias_mV,
    y=P_out_dBm,
    z=dGexp_G0 / GN_G0,
    xlabel="<i>V</i> (mV)",
    ylabel="<i>P</i> (dBm)",
    zlabel="d<i>I</i>/d<i>V</i> (G<sub>N</sub>)",
    zlim=(0.3, 2.5),
    name=title,
    dataset=dataset,
)

from superconductivity.visuals.stl import write_3D_print

write_3D_print(
    z=np.clip(dGexp_G0 / GN_G0, 0.3, 2.5),
    title=title,
    dataset=dataset,
)

from superconductivity.visuals.svg import save_axis

axis = save_axis(
    values=Vbias_mV,
    label=r"$V\ (\mathrm{mV})$",
    title=f"x {title}",
    dataset=dataset,
)
axis = save_axis(
    values=P_out_dBm,
    label=r"$P_\mathrm{out}\ (\mathrm{dBm})$",
    title=f"y {title}",
    dataset=dataset,
    vertical=True,
)
axis = save_axis(
    values=np.clip(dGexp_G0 / GN_G0, 0.3, 2.5),
    label=r"$\mathrm{d}I/\mathrm{d}V\ (G_\mathrm{N})$",
    title=f"z {title}",
    dataset=dataset,
)
axis = save_axis(
    values=np.clip(dGexp_G0 / GN_G0, 0.3, 2.5),
    label=r"$\mathrm{d}I/\mathrm{d}V\ (G_\mathrm{N})$",
    title=f"z {title}2",
    dataset=dataset,
    vertical=True,
)

from superconductivity.visuals.luanti import export_dataset

export_dataset(
    dGexp_G0 / GN_G0,
    title=title,
    dataset=dataset,
    zlim=(0.3, 2.5),
    hmin=0,
    hmax=120,
    ylen=124,
    xlen=500,
)

PosixPath('visuals/datasets/6 dGexp/map.u16le')

In [618]:
title = "7 dRexp"
dataset = "visuals"

from superconductivity.visuals.plotly import get_all

importlib.reload(sys.modules["superconductivity.visuals.plotly"])

fig_surface, fig_slider, fig_heatmap = get_all(
    x=Ibias_nA,
    y=P_out_dBm,
    z=dRexp_R0 * GN_G0,
    xlabel="<i>I</i> (nA)",
    ylabel="<i>P</i> (dBm)",
    zlabel="d<i>V</i>/d<i>I</i> (R<sub>N</sub>)",
    zlim=(0.3, 1.5),
    name=title,
    dataset=dataset,
)

from superconductivity.visuals.stl import write_3D_print

write_3D_print(
    z=np.clip(dRexp_R0 * GN_G0, 0.3, 1.5),
    title=title,
    dataset=dataset,
)

from superconductivity.visuals.svg import save_axis

axis = save_axis(
    values=Ibias_nA,
    label=r"$I\ (\mathrm{nA})$",
    title=f"x {title}",
    dataset=dataset,
)
axis = save_axis(
    values=P_out_dBm,
    label=r"$P_\mathrm{out}\ (\mathrm{dBm})$",
    title=f"y {title}",
    dataset=dataset,
    vertical=True,
)
axis = save_axis(
    values=np.clip(dRexp_R0 * GN_G0, 0.3, 1.5),
    label=r"$\mathrm{d}V/\mathrm{d}I\ (R_\mathrm{N})$",
    title=f"z {title}",
    dataset=dataset,
)
axis = save_axis(
    values=np.clip(dRexp_R0 * GN_G0, 0.3, 1.5),
    label=r"$\mathrm{d}V/\mathrm{d}I\ (R_\mathrm{N})$",
    title=f"z {title}2",
    dataset=dataset,
    vertical=True,
)

from superconductivity.visuals.luanti import export_dataset

export_dataset(
    dRexp_R0 * GN_G0,
    title=title,
    dataset=dataset,
    zlim=(0.3, 1.5),
    hmin=0,
    hmax=120,
    ylen=124,
    xlen=500,
)

PosixPath('visuals/datasets/7 dRexp/map.u16le')

In [619]:
# build worlds and html

dataset = "visuals"

from superconductivity.visuals.plotly import build_html

build_html(
    dataset=dataset,
    title="main",
    title_html="Atomic Contact (<i>G=1.97 G<sub>0</sub></i>, <i>ν=15 </i>GHz, <i>τ={0.78, 0.66, 0.31, 0.16, 0.08, 0.03}</i>)",
)

from superconductivity.visuals.luanti import build_worlds

build_worlds(
    prefix="AC (1.97G0, 15GHz) ",
    dataset=dataset,
)

built world: AC (1.97G0, 15GHz) 4_Iexp
built world: AC (1.97G0, 15GHz) 5_Vexp
built world: AC (1.97G0, 15GHz) 6_dGexp
built world: AC (1.97G0, 15GHz) 7_dRexp
Copying worlds from: /Users/oliver/Documents/superconductivity/evaluation/AC 2.09G0 15GHz (PAMAR)/visuals/worlds
Deployed mods and worlds to:
  /Users/oliver/Library/Application Support/minetest


python(78563) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


['AC (1.97G0, 15GHz) 4_Iexp',
 'AC (1.97G0, 15GHz) 5_Vexp',
 'AC (1.97G0, 15GHz) 6_dGexp',
 'AC (1.97G0, 15GHz) 7_dRexp']